# B92 Algorithm

## Introduction

The **B92 protocol**, proposed by Charles Bennett in 1992, is a quantum key distribution (QKD) scheme that allows two parties, Alice and Bob, to establish a shared secret key over a public quantum channel.

Unlike BB84, which uses four states, B92 uses only **two non-orthogonal quantum states** to encode classical bits:

- Bit `0` is encoded as `|0>` (the computational basis state)
- Bit `1` is encoded as `|+>` (the Hadamard basis state: equal superposition of |0> and |1>)

The security of B92 relies on a fundamental property of quantum mechanics: **non-orthogonal states cannot be perfectly distinguished**. Any eavesdropper (Eve) trying to measure the qubits will inevitably disturb the quantum states and introduce detectable errors.

### Protocol outline

1. **Alice** generates a random bit string and encodes each bit as a qubit using the two non-orthogonal states above.
2. **Bob** randomly chooses a measurement basis (Z or X) for each qubit.
3. Bob keeps only the results where he measured `|1>` -- these cases guarantee that his basis was incompatible with Alice's encoding, so the bits are deterministic and unambiguous.
4. Bob announces the positions where he got `|1>`. Alice and Bob use those positions to build the shared key.
5. A subset of the key bits can be compared publicly to detect eavesdropping.

The resulting key is information-theoretically secure as long as the quantum channel is authenticated.

### Imports

We use **SymPy** for symbolic quantum computation (`qapply`, `HadamardGate`, `Qubit`) and Python's `random` module to generate classical random bits. The `measure_all_oneshot` utility performs a single projective measurement on a qubit state, collapsing it to a basis state according to its probability amplitudes.

In [16]:
from sympy.physics.quantum import qapply
from sympy.physics.quantum.gate import HadamardGate
from sympy.physics.quantum.qubit import Qubit
from random import getrandbits

from util.measure_all import measure_all_oneshot
from util.util import zero

## Step 1 -- Alice's Side

Alice starts by generating `N` random classical bits. Each bit will later be mapped to one of two non-orthogonal quantum states before being sent to Bob over the quantum channel.

In [17]:
def get_alice_bits(size):
    """Generate a list of `size` random classical bits for Alice.

    :param size: number of bits to generate
    :return: list of random bits (each 0 or 1)
    """
    return [getrandbits(1) for _ in range(size)]


N = 32
alice_bits = get_alice_bits(N)
print(f"Alice random bits = {alice_bits}")

Alice random bits = [1, 0, 1, 1, 1, 1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 0, 1, 1, 0, 0, 0, 1]


### Alice's encoding

Alice maps each classical bit to a qubit using a fixed, **non-orthogonal** pair of states:

- Alice bit **0** is sent as state **|0>**, the Z-basis ground state.
- Alice bit **1** is sent as state **|+> = (|0> + |1>) / sqrt(2)**, the X-basis +1 eigenstate.

These two states are not orthogonal (`<0|+> = 1/sqrt(2) != 0`), which is what makes them impossible to distinguish perfectly. Applying `H` to `|0>` produces `|+>`, so in code bit `1` is encoded by applying a `HadamardGate` to the `|0>` state.

In [18]:
def b92_alice_encoding(alice_bits):
    """Encode Alice's classical bits as non-orthogonal qubits.

    Encoding rule (non-orthogonal basis {|0>, |+>}):
        bit 0 -> |0>  (Z-basis ground state, represented by `zero`)
        bit 1 -> |+>  (X-basis +1 eigenstate, obtained via H|0>)

    :param alice_bits: list of classical bits (0 or 1)
    :return: list of encoded qubit states
    """
    plus_state = qapply(HadamardGate(0) * Qubit(0))  # |+> = H|0>
    return [plus_state if b == 1 else zero for b in alice_bits]


alice_encoded = b92_alice_encoding(alice_bits)
print(f"Alice encoded: {alice_encoded}")

Alice encoded: [sqrt(2)*|0>/2 + sqrt(2)*|1>/2, |0>, sqrt(2)*|0>/2 + sqrt(2)*|1>/2, sqrt(2)*|0>/2 + sqrt(2)*|1>/2, sqrt(2)*|0>/2 + sqrt(2)*|1>/2, sqrt(2)*|0>/2 + sqrt(2)*|1>/2, |0>, |0>, sqrt(2)*|0>/2 + sqrt(2)*|1>/2, |0>, sqrt(2)*|0>/2 + sqrt(2)*|1>/2, |0>, |0>, sqrt(2)*|0>/2 + sqrt(2)*|1>/2, |0>, |0>, sqrt(2)*|0>/2 + sqrt(2)*|1>/2, |0>, |0>, |0>, |0>, sqrt(2)*|0>/2 + sqrt(2)*|1>/2, sqrt(2)*|0>/2 + sqrt(2)*|1>/2, |0>, sqrt(2)*|0>/2 + sqrt(2)*|1>/2, |0>, sqrt(2)*|0>/2 + sqrt(2)*|1>/2, sqrt(2)*|0>/2 + sqrt(2)*|1>/2, |0>, |0>, |0>, sqrt(2)*|0>/2 + sqrt(2)*|1>/2]


## Step 2 -- Bob's Side


Bob independently generates `N` random classical bits. These bits determine which measurement basis he will use for each qubit he receives from Alice.

In [19]:
def get_bob_bits(size):
    """Generate a list of `size` random classical bits for Bob.

    :param size: number of bits to generate
    :return: list of random bits (each 0 or 1)
    """
    return [getrandbits(1) for _ in range(size)]


bob_bits = get_bob_bits(N)
print(f"Bob random bits = {bob_bits}")

Bob random bits = [1, 0, 1, 0, 1, 1, 1, 1, 0, 1, 0, 0, 1, 1, 0, 1, 0, 1, 0, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1]


### Bob's basis selection

Bob maps each of his random bits to a measurement basis:

- Bob bit `0` -> measure in the **Z basis** (`{|0>, |1>}`)
- Bob bit `1` -> measure in the **X basis** (`{|+>, |->}`)

Neither basis can perfectly distinguish Alice's two states, which is exactly what creates the key-extraction asymmetry exploited by the protocol.

In [20]:
def b92_get_bob_basis(bits):
    """Convert Bob's random bits to measurement basis labels.

    Convention:
        bit 0 -> 'Z'  (computational basis)
        bit 1 -> 'X'  (Hadamard basis)

    :param bits: list of classical bits (0 or 1)
    :return: list of basis labels ('Z' or 'X')
    """
    return ["Z" if b == 0 else "X" for b in bits]


bob_basis = b92_get_bob_basis(bob_bits)
print(f"Bob basis: {bob_basis}")

Bob basis: ['X', 'Z', 'X', 'Z', 'X', 'X', 'X', 'X', 'Z', 'X', 'Z', 'Z', 'X', 'X', 'Z', 'X', 'Z', 'X', 'Z', 'X', 'X', 'Z', 'Z', 'X', 'Z', 'X', 'X', 'Z', 'X', 'Z', 'Z', 'X']


### Bob's measurement

For each received qubit, Bob applies a basis change if necessary and then performs a projective measurement:

- If he chose the **Z basis**, he measures directly.
- If he chose the **X basis**, he first applies a Hadamard gate to rotate from the X basis to the Z basis, then measures.

The outcome is either `|0>` or `|1>`. Only the `|1>` outcomes are useful -- they provide unambiguous information about Alice's bit (see the analysis table in the next section).

In [21]:
def b92_bob_measure(bob_basis, alice_encoded):
    """Measure Alice's qubits in Bob's chosen bases.

    For each qubit:
        - Z basis: measure directly in the computational basis.
        - X basis: apply H first to rotate to computational basis, then measure.

    :param bob_basis: list of basis labels ('Z' or 'X') chosen by Bob
    :param alice_encoded: list of qubit states sent by Alice
    :return: list of measurement outcomes (Qubit(0) or Qubit(1))
    """
    measures = []
    for qubit, basis in zip(alice_encoded, bob_basis):
        if basis == "X":
            # Rotate from X basis to Z basis before measuring
            qubit = qapply(HadamardGate(0) * qubit)  # H|q>
        measures.append(measure_all_oneshot(qubit))
    return measures


bob_measures = b92_bob_measure(bob_basis, alice_encoded)
print(f"Bob measures: {bob_measures}")

Bob measures: [|0>, |0>, |0>, |1>, |0>, |0>, |1>, |1>, |1>, |0>, |1>, |0>, |0>, |0>, |0>, |1>, |1>, |0>, |0>, |1>, |0>, |0>, |1>, |1>, |0>, |0>, |0>, |0>, |1>, |0>, |0>, |0>]


## Step 3 -- Key Sifting

### Analysis of all possible (Alice bit, Bob basis) combinations

Every combination of Alice's encoding and Bob's basis, with outcome probabilities:

- Alice bit $0$ sends $|0\rangle$, Bob basis Z: outcome $|0\rangle$ with prob. $1$ -- outcome 0, **not useful**
- Alice bit $1$ sends $|{+}\rangle$, Bob basis Z: outcome $|0\rangle$ with prob. $\tfrac{1}{2}$ -- outcome 0, **not useful**
- Alice bit $1$ sends $|{+}\rangle$, Bob basis Z: outcome $|1\rangle$ with prob. $\tfrac{1}{2}$ -- outcome 1, **not useful** (ambiguous)
- Alice bit $0$ sends $|0\rangle$, Bob basis X: outcome $|{+}\rangle$ with prob. $\tfrac{1}{2}$ -- outcome 0, **not useful**
- Alice bit $0$ sends $|0\rangle$, Bob basis X: outcome $|{-}\rangle$ with prob. $\tfrac{1}{2}$ -- outcome **1**, **useful!**
- Alice bit $1$ sends $|{+}\rangle$, Bob basis X: outcome $|{+}\rangle$ with prob. $1$ -- outcome 0, **not useful**

**Key observation:** Bob can only measure outcome $1$ when Alice sent $|0\rangle$ and Bob used the X basis -- never the other way around. This means:

- Whenever Bob gets outcome `1`, he knows with **certainty** that Alice's bit was `0`.
- Outcome `0` is ambiguous: it can result from any combination, so those positions are discarded.

Both parties therefore derive the same key bit from every position where Bob measured `1`: Bob contributes his bit (`1`) and Alice contributes `1 - alice_bit` (which equals `1` when `alice_bit = 0`). Only these unambiguous positions form the final shared key.

Bob broadcasts over the public classical channel the list of positions where he obtained outcome `1`. Alice and Bob will use only those positions to construct the shared key.

In [22]:
def find_ones(bob_measures):
    """Return the indices of positions where Bob measured outcome |1>.

    These are the only positions that carry unambiguous key information
    in the B92 protocol.

    :param bob_measures: list of measurement outcomes (Qubit states)
    :return: list of integer indices where the outcome was Qubit(1)
    """
    return [i for i, outcome in enumerate(bob_measures) if outcome == Qubit(1)]


bob_one_indices = find_ones(bob_measures)
print(f"bob_one_indices={bob_one_indices}")

bob_one_indices=[3, 6, 7, 8, 10, 15, 16, 19, 22, 23, 28]


### Key generation

Using the sifted positions (where Bob measured `1`), both parties extract their key bits independently -- no further quantum communication is needed:

- **Alice's key bit** at position `i` = `1 - alice_bits[i]`. Because at those positions Alice's bit was always `0`, this flips it to `1`, matching Bob's measurement.
- **Bob's key bit** at position `i` = `bob_bits[i]`, which is always `1` at those positions.

Both sides therefore produce the same key without ever transmitting it.

In [23]:
def b92_generate_keys(alice_bits, bob_bits, sifted_indices):
    """Build the shared key from the sifted positions.

    At each sifted position i (where Bob measured |1>):
        - Alice knows her bit was 0, so her key bit = 1 - alice_bits[i] = 1.
        - Bob's key bit = bob_bits[i] = 1 (only kept positions where he got 1).
    Both bits are therefore always equal.

    :param alice_bits: full list of Alice's random bits
    :param bob_bits: full list of Bob's random bits
    :param sifted_indices: positions where Bob measured outcome |1>
    :return: tuple (alice_key, bob_key) -- both lists are identical
    """
    alice_key = [1 - alice_bits[i] for i in sifted_indices]
    bob_key = [bob_bits[i] for i in sifted_indices]
    return alice_key, bob_key


alice_key, bob_key = b92_generate_keys(alice_bits, bob_bits, bob_one_indices)
print(f"bob key  = {bob_key}")
print(f"alice key= {alice_key}")

bob key  = [0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 1]
alice key= [0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 1]


### Verification

As a sanity check we assert that Alice's and Bob's keys are identical. In a real deployment, only a random subset of key bits would be compared publicly (and then discarded) to detect eavesdropping; the remaining bits form the secret key.

In [24]:
assert alice_key == bob_key

## Tests

In [25]:
def test_0():
    """Standard run: verify that keys match for a fixed bit string."""
    alice_bits = [0, 1, 1, 0, 1, 1, 1, 1, 0, 1, 0, 1,
                  1, 0, 0, 1, 1, 1, 0, 1, 0, 1, 1, 0,
                  0, 1, 0, 1, 1, 0, 1, 1]
    bob_bits = [1, 0, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0,
                0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 1,
                1, 1, 1, 1, 1, 1, 1, 1]

    # Bob measured |1> at these positions (pre-computed for this fixed input)
    sifted = [1, 2, 7, 12, 16, 23]
    alice_key, bob_key = b92_generate_keys(alice_bits, bob_bits, sifted)
    print(f"test_0: bob key={bob_key}, alice key={alice_key}")
    assert alice_key == bob_key


test_0()

test_0: bob key=[0, 0, 0, 0, 0, 1], alice key=[0, 0, 0, 0, 0, 1]


In [26]:
def test_1():
    """Empty sifted set: both keys must be empty lists."""
    alice_bits = [0, 1, 0, 1]
    bob_bits = [0, 0, 0, 0]
    sifted = []  # Bob never measured |1>
    alice_key, bob_key = b92_generate_keys(alice_bits, bob_bits, sifted)
    print(f"test_1: bob key={bob_key}, alice key={alice_key}")
    assert alice_key == []
    assert bob_key == []


test_1()

test_1: bob key=[], alice key=[]


In [27]:
def test_2():
    """All-zeros Alice input: encoding always produces |0> qubits."""
    all_zero_bits = [0] * 8
    encoded = b92_alice_encoding(all_zero_bits)
    # Every encoded qubit must be the |0> state
    assert all(q == zero for q in encoded), "Expected all |0> states"
    print("test_2: all-zero encoding OK")


test_2()

test_2: all-zero encoding OK


In [28]:
def test_3():
    """All-ones Alice input: encoding always produces |+> qubits."""
    all_one_bits = [1] * 8
    encoded = b92_alice_encoding(all_one_bits)
    plus_state = qapply(HadamardGate(0) * Qubit(0))
    # Every encoded qubit must be the |+> state
    assert all(q == plus_state for q in encoded), "Expected all |+> states"
    print("test_3: all-ones encoding OK")


test_3()

test_3: all-ones encoding OK


In [29]:
def test_4():
    """Basis mapping: bit 0 -> Z, bit 1 -> X."""
    bits = [0, 1, 0, 1, 0]
    expected = ["Z", "X", "Z", "X", "Z"]
    assert b92_get_bob_basis(bits) == expected
    print("test_4: basis mapping OK")


test_4()

test_4: basis mapping OK


In [30]:
def test_5():
    """find_ones returns correct indices for a known measurement list."""
    # Build a synthetic measurement list: positions 1 and 3 are |1>
    outcomes = [Qubit(0), Qubit(1), Qubit(0), Qubit(1), Qubit(0)]
    assert find_ones(outcomes) == [1, 3]
    print("test_5: find_ones OK")


test_5()

test_5: find_ones OK
